# Analyze CIFAR ResNet Head Experiments

This notebook visualizes the clean real-world analogue of the synthetic support experiments.

We analyze the first MLP-head layer trained on frozen ResNet embeddings using:

- test accuracy
- weight matrix heatmaps
- Gram matrix heatmaps
- singular value spectra
- Gram eigenvalue spectra
- stable rank
- effective rank
- sorted column norms

The central question is:

> Do different optimizers make the first MLP-head layer collapse onto a low-dimensional feature subspace?


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

RESULTS_DIR = Path("results_clean")

RUNS = [
    "gd",
    "sgd512",
    "sgd128",
    "sgd32",
    "gd_wd01",
    "adam",
    "adamw",
    "muon128",
]

LABELS = {
    "gd": "GD",
    "sgd512": "SGD bs=512",
    "sgd128": "SGD bs=128",
    "sgd32": "SGD bs=32",
    "gd_wd01": "GD + WD",
    "adam": "Adam",
    "adamw": "AdamW",
    "muon128": "Muon",
}

def load_run(run):
    path = RESULTS_DIR / f"{run}.npz"
    if not path.exists():
        raise FileNotFoundError(f"Missing result file: {path}")
    return np.load(path, allow_pickle=True)

def metric(data, name):
    return data[f"metric_{name}"]

def checkpoint_weight(data, layer=0, which="final"):
    keys = [k for k in data.files if k.endswith(f"_layer{layer}_weights")]
    keys = sorted(keys, key=lambda k: int(k.split("_")[0].replace("step", "")))
    if which == "init":
        return data[keys[0]]
    if which == "final":
        return data[keys[-1]]
    raise ValueError("which must be 'init' or 'final'")

def singular_values(W):
    return np.linalg.svd(W, compute_uv=False)

def effective_rank_from_singular_values(s, eps=1e-12):
    s = s[s > eps]
    if len(s) == 0:
        return 0.0
    p = s / s.sum()
    return float(np.exp(-(p * np.log(p + eps)).sum()))

def stable_rank_from_singular_values(s, eps=1e-12):
    if len(s) == 0:
        return 0.0
    return float(np.sum(s ** 2) / (s[0] ** 2 + eps))

def stable_rank(W):
    return stable_rank_from_singular_values(singular_values(W))

def effective_rank(W):
    return effective_rank_from_singular_values(singular_values(W))

def right_gram(W):
    return W.T @ W

def left_gram(W):
    return W @ W.T

def sorted_eigs_symmetric(A):
    return np.linalg.eigvalsh(A)[::-1]


## 1. Sanity check: which runs are available?

In [ ]:
available = []
missing = []

for run in RUNS:
    if (RESULTS_DIR / f"{run}.npz").exists():
        available.append(run)
    else:
        missing.append(run)

print("Available runs:", available)
print("Missing runs:", missing)

RUNS = available


## 2. Test accuracy over optimizer steps

In [ ]:
plt.figure(figsize=(8, 4))

for run in RUNS:
    data = load_run(run)
    steps = metric(data, "step")
    acc = metric(data, "test_acc")
    plt.plot(steps, acc, label=LABELS.get(run, run))

plt.xlabel("optimizer step")
plt.ylabel("test accuracy")
plt.title("Test accuracy vs optimizer step")
plt.grid(True)
plt.legend()
plt.show()


## 3. Stable rank of the first MLP layer

Stable rank:

\[
\operatorname{srank}(W) = \frac{\|W\|_F^2}{\|W\|_2^2}
\]

This is often more sensitive to spectral concentration than entropy-based effective rank.


In [ ]:
plt.figure(figsize=(8, 4))

for run in RUNS:
    data = load_run(run)
    steps = metric(data, "step")
    sr = metric(data, "layer0_stable_rank")
    plt.plot(steps, sr, label=LABELS.get(run, run))

plt.xlabel("optimizer step")
plt.ylabel("stable rank")
plt.title("Layer 0 stable rank vs optimizer step")
plt.grid(True)
plt.legend()
plt.show()


## 4. Effective rank of the first MLP layer

In [ ]:
plt.figure(figsize=(8, 4))

for run in RUNS:
    data = load_run(run)
    steps = metric(data, "step")
    er = metric(data, "layer0_eff_rank")
    plt.plot(steps, er, label=LABELS.get(run, run))

plt.xlabel("optimizer step")
plt.ylabel("effective rank")
plt.title("Layer 0 effective rank vs optimizer step")
plt.grid(True)
plt.legend()
plt.show()


## 5. Per-layer stable rank

This lets us see whether collapse happens mainly in the first head layer or throughout the MLP.


In [ ]:
if RUNS:
    data0 = load_run(RUNS[0])
    n_layers = 0
    while f"metric_layer{n_layers}_stable_rank" in data0.files:
        n_layers += 1
else:
    n_layers = 0

fig, axes = plt.subplots(1, n_layers, figsize=(5 * n_layers, 4), squeeze=False)
axes = axes[0]

for layer in range(n_layers):
    ax = axes[layer]
    for run in RUNS:
        data = load_run(run)
        steps = metric(data, "step")
        sr = metric(data, f"layer{layer}_stable_rank")
        ax.plot(steps, sr, label=LABELS.get(run, run))
    ax.set_title(f"Layer {layer} stable rank")
    ax.set_xlabel("optimizer step")
    ax.set_ylabel("stable rank")
    ax.grid(True)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


## 6. Final singular value spectra of \(W_1\)

In [ ]:
fig, axes = plt.subplots(len(RUNS), 1, figsize=(8, 3 * len(RUNS)), squeeze=False)
axes = axes[:, 0]

for ax, run in zip(axes, RUNS):
    data = load_run(run)
    W = checkpoint_weight(data, layer=0, which="final")
    s = singular_values(W)

    ax.bar(np.arange(1, len(s) + 1), s)
    ax.set_title(f"{LABELS.get(run, run)} — final singular values of W1")
    ax.set_xlabel("index")
    ax.set_ylabel("singular value")
    ax.grid(True, axis="y")

plt.tight_layout()
plt.show()


## 7. Final eigenvalue spectra of the right Gram matrix \(W_1^\top W_1\)

The right Gram matrix lives in input-feature space. Its eigenvalue spectrum shows whether the layer uses a low-dimensional subspace of the frozen ResNet embeddings.


In [ ]:
fig, axes = plt.subplots(len(RUNS), 1, figsize=(8, 3 * len(RUNS)), squeeze=False)
axes = axes[:, 0]

for ax, run in zip(axes, RUNS):
    data = load_run(run)
    W = checkpoint_weight(data, layer=0, which="final")
    eigs = sorted_eigs_symmetric(right_gram(W))

    ax.bar(np.arange(1, len(eigs) + 1), eigs)
    ax.set_title(f"{LABELS.get(run, run)} — eigenvalues of W1.T @ W1")
    ax.set_xlabel("index")
    ax.set_ylabel("eigenvalue")
    ax.grid(True, axis="y")

plt.tight_layout()
plt.show()


## 8. Top-64 Gram eigenvalues

This is often easier to read than plotting all 512 eigenvalues.


In [ ]:
TOP = 64

plt.figure(figsize=(9, 5))

for run in RUNS:
    data = load_run(run)
    W = checkpoint_weight(data, layer=0, which="final")
    eigs = sorted_eigs_symmetric(right_gram(W))[:TOP]
    eigs = eigs / (eigs[0] + 1e-12)
    plt.plot(np.arange(1, TOP + 1), eigs, label=LABELS.get(run, run))

plt.xlabel("eigenvalue rank")
plt.ylabel("normalized eigenvalue")
plt.title("Top Gram eigenvalues of W1.T @ W1")
plt.grid(True)
plt.legend()
plt.show()


## 9. Sorted column norms of \(W_1\)

Column norms are the closest real-data analogue of “feature usage per input coordinate.”

Unlike the synthetic setup, these are not true relevance labels. They only show coordinate concentration in the ResNet embedding basis.


In [ ]:
plt.figure(figsize=(9, 5))

for run in RUNS:
    data = load_run(run)
    W = checkpoint_weight(data, layer=0, which="final")
    norms = np.linalg.norm(W, axis=0)
    norms = np.sort(norms)[::-1]
    norms = norms / (norms[0] + 1e-12)
    plt.plot(np.arange(1, len(norms) + 1), norms, label=LABELS.get(run, run))

plt.xlabel("feature rank")
plt.ylabel("normalized column norm")
plt.title("Sorted W1 column norms")
plt.grid(True)
plt.legend()
plt.show()


## 10. Weight matrix heatmaps

Columns are sorted by final column norm so the strongest input coordinates appear on the left.


In [ ]:
fig, axes = plt.subplots(len(RUNS), 1, figsize=(9, 3 * len(RUNS)), squeeze=False)
axes = axes[:, 0]

for ax, run in zip(axes, RUNS):
    data = load_run(run)
    W = checkpoint_weight(data, layer=0, which="final")

    col_norms = np.linalg.norm(W, axis=0)
    order = np.argsort(col_norms)[::-1]
    W_sorted = W[:, order]

    vmax = np.abs(W_sorted).max()
    im = ax.imshow(W_sorted, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_title(f"{LABELS.get(run, run)} — final W1, columns sorted by norm")
    ax.set_xlabel("input embedding coordinate rank")
    ax.set_ylabel("hidden unit")

plt.tight_layout()
plt.show()


## 11. Right Gram matrix heatmaps

These visualize pairwise interactions between input embedding directions after the first MLP layer.


In [ ]:
fig, axes = plt.subplots(len(RUNS), 1, figsize=(7, 4 * len(RUNS)), squeeze=False)
axes = axes[:, 0]

for ax, run in zip(axes, RUNS):
    data = load_run(run)
    W = checkpoint_weight(data, layer=0, which="final")

    col_norms = np.linalg.norm(W, axis=0)
    order = np.argsort(col_norms)[::-1]
    W_sorted = W[:, order]

    G = right_gram(W_sorted)

    im = ax.imshow(G, aspect="auto", cmap="viridis")
    ax.set_title(f"{LABELS.get(run, run)} — final W1.T @ W1")
    ax.set_xlabel("input coordinate rank")
    ax.set_ylabel("input coordinate rank")

plt.tight_layout()
plt.show()


## 12. Init vs final Gram eigenvalue spectra for each run

In [ ]:
fig, axes = plt.subplots(len(RUNS), 2, figsize=(12, 3 * len(RUNS)), squeeze=False)

for row, run in enumerate(RUNS):
    data = load_run(run)

    for col, which in enumerate(["init", "final"]):
        W = checkpoint_weight(data, layer=0, which=which)
        eigs = sorted_eigs_symmetric(right_gram(W))[:64]
        ax = axes[row, col]
        ax.bar(np.arange(1, len(eigs) + 1), eigs)
        ax.set_title(f"{LABELS.get(run, run)} — {which}")
        ax.set_xlabel("eigenvalue rank")
        ax.set_ylabel("eigenvalue")
        ax.grid(True, axis="y")

plt.tight_layout()
plt.show()


## 13. Summary table

In [ ]:
rows = []

for run in RUNS:
    data = load_run(run)
    W = checkpoint_weight(data, layer=0, which="final")
    s = singular_values(W)

    rows.append({
        "run": run,
        "test_acc": metric(data, "test_acc")[-1],
        "stable_rank_W1": stable_rank_from_singular_values(s),
        "effective_rank_W1": effective_rank_from_singular_values(s),
        "frobenius_norm_W1": np.linalg.norm(W, "fro"),
        "spectral_norm_W1": s[0],
    })

print(f"{'run':12s} {'acc':>8s} {'stable_rank':>14s} {'eff_rank':>12s} {'frob':>12s} {'spectral':>12s}")
print("-" * 78)

for r in rows:
    print(
        f"{r['run']:12s} "
        f"{r['test_acc']:8.4f} "
        f"{r['stable_rank_W1']:14.2f} "
        f"{r['effective_rank_W1']:12.2f} "
        f"{r['frobenius_norm_W1']:12.4f} "
        f"{r['spectral_norm_W1']:12.4f}"
    )
